In [1]:
import os

from pathlib import Path
import pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
D = '/content/drive/MyDrive/MinneMUDAC 26 for BU/'

fs    = pd.read_excel(D + 'By Organizers/Data/org_FSStats.xlsx')
orgs  = pd.read_excel(D + 'By Organizers/Data/Organization.xlsx')
mb    = pd.read_excel(D + 'By Organizers/Data/marketbucksUsage.xlsx')
calls = pd.read_excel(D + 'By Organizers/Data/Incoming Call.xlsx')
mmg   = pd.read_excel(D+ 'MMG_County_MN_2021_2024.xlsx')

fs.shape, orgs.shape, mb.shape, calls.shape, mmg.shape

((28287, 58), (1202, 68), (13573, 19), (49727, 31), (348, 26))

In [3]:
fs.info(), orgs.info(), mb.info(), calls.info(), mmg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28287 entries, 0 to 28286
Data columns (total 58 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   __ID                 28287 non-null  int64         
 1   _id_site             28287 non-null  float64       
 2   _Year                28287 non-null  int64         
 3   _Month               28287 non-null  int64         
 4   DateForSearch        28287 non-null  datetime64[ns]
 5   zzCreationTS         28287 non-null  object        
 6   zzModificationTS     28287 non-null  object        
 7   AdultsReg            26225 non-null  float64       
 8   ChildrenReg          26101 non-null  float64       
 9   HouseholdsReg        26049 non-null  float64       
 10  IndividualsReg       26260 non-null  float64       
 11  Meals                12806 non-null  float64       
 12  Misc                 0 non-null      float64       
 13  People               0 non-null

(None, None, None, None, None)

In [4]:
empty = fs.columns[fs.notna().sum() == 0]
print(len(empty), list(empty))

fs = fs.drop(columns=empty)
fs.shape

24 ['Misc', 'People', 'Snacks', 'ChildrenExtra', 'ChildrenProduce', 'ChildrenHoliday', 'AdultsExtra', 'AdultsProduce', 'AdultsHoliday', 'IndividualsExtra', 'SeniorsExtra', 'SeniorsProduce', 'SeniorsHoliday', 'IndividualsProduce', 'IndividualsHoliday', 'UniqueVisitsExtra', 'UniqueVisitsProduce', 'UniqueVisitsHoliday', 'HouseholdsExtra', 'HouseholdsProduce', 'HouseholdsHoliday', 'PoundsExtra', 'PoundsProduce', 'PoundsHoliday']


(28287, 34)

In [5]:
fs = fs.drop(columns=['SuperShelf'])          # broken, 1 site
orgs.SuperShelfStatus.value_counts(dropna=False)   # the usable one, 84 orgs

,count
SuperShelfStatus,
NaN,1118
Certified,63
In Process,20
1,1


In [6]:
mmg.columns = [str(c).strip() for c in mmg.columns]
mmg['fips5'] = mmg.FIPS.astype(str).str.zfill(5)
mn = mmg[mmg.fips5.str[:2] == '27']

mn.groupby('Year').size()                                     # want 87 per year
mn.groupby('Year')['# of Food Insecure Persons Overall'].sum() # 2023 -> 581270, 2024 -> 636610
mn.groupby('Year')['Overall Food Insecurity Rate'].median()    # 2023 -> 0.111, 2024 -> 0.123
mmg.FIPS.dtype                                                # int loses leading zeros for other states
sorted(mmg.columns)

['# of Food Insecure Children',
 '# of Food Insecure Persons Overall',
 '% FI > SNAP Threshold',
 '% FI ≤ SNAP Threshold',
 '% food insecure children in HH w/ HH incomes above 185 FPL',
 '% food insecure children in HH w/ HH incomes below 185 FPL',
 'Census Division',
 'Census Region',
 'Child Food Insecurity Rate',
 'Cost Per Meal',
 'County, State',
 'FIPS',
 'FNS Region',
 'Food Insecurity Rate among Black Persons (all ethnicities)',
 'Food Insecurity Rate among Hispanic Persons (any race)',
 'Food Insecurity Rate among White, non-Hispanic Persons',
 'Imputed Market Basket Data',
 'Overall Food Insecurity Rate',
 'Ratio (1 in X) Overall',
 'Rural-Urban Continuum Code (2013)',
 'Rural-Urban Continuum Code (2023)',
 'SNAP Threshold',
 'State',
 'Weighted Annual Food Budget Shortfall',
 'Weighted weekly $ needed by FI',
 'Year',
 'fips5']

In [7]:
mn.groupby('Year').size()
mn.groupby('Year')['# of Food Insecure Persons Overall'].sum()
# mn.groupby('Year')['Overall Food Insecurity Rate'].median()

,# of Food Insecure Persons Overall
Year,
2021,342640
2022,509450
2023,581270
2024,636610


In [8]:
fs[(fs._Year.isin([2023,2024])) & (~fs._Month.between(1,12))][
    ['_id_site','_Year','_Month','DateForSearch','County','PoundsReg']]

,_id_site,_Year,_Month,DateForSearch,County,PoundsReg
10251,1148.0,2023,15,2024-03-01,Polk,2043.0
12369,14810.0,2023,0,2022-12-01,St. Louis,1785.0
12370,1072.0,2023,0,2022-12-01,St. Louis,8778.0
12371,14717.0,2023,0,2022-12-01,Carlton,11625.0
12372,1103.0,2023,0,2022-12-01,St. Louis,12128.0
12373,1088.0,2023,0,2022-12-01,St. Louis,40515.0
12374,1110.0,2023,0,2022-12-01,Carlton,13535.0
12375,1108.0,2023,0,2022-12-01,St. Louis,5820.0
12376,1250.0,2023,0,2022-12-01,St. Louis,6948.0
12377,1410.0,2023,0,2022-12-01,St. Louis,37695.0


In [9]:
fs['dt'] = pd.to_datetime(fs.DateForSearch)
fs2324 = fs[fs.dt.dt.year.isin([2023,2024])].copy()   # 12,772 rows

In [10]:
fs2324.groupby(fs2324.dt.dt.year)[['HouseholdsReg','IndividualsReg','PoundsReg']].sum()

,HouseholdsReg,IndividualsReg,PoundsReg
dt,,,
2023,2790625.0,7743201.0,1.414514e+08
2024,2918714.0,9146694.0,1.578168e+08


In [11]:
fs2324 = (fs2324
    .sort_values(['_id_site','dt','IndividualsReg','PoundsReg','__ID'],
                 ascending=[True,True,False,False,False])
    .drop_duplicates(['_id_site','dt'], keep='first')
    .copy())

bad = fs2324.HouseholdsReg > fs2324.IndividualsReg   # households can't exceed individuals
print(fs2324.shape, bad.sum())                       # (12741, 34)  53

(12741, 34) 53


In [12]:
clean = fs2324[~bad]
s = clean.groupby('_id_site')[['IndividualsReg','HouseholdsReg']].sum()
site_ratio = s.IndividualsReg / s.HouseholdsReg

bad_tbl = fs2324[bad].copy()
bad_tbl['site_ratio'] = bad_tbl._id_site.map(site_ratio).round(2)
bad_tbl['overshoot']  = (bad_tbl.HouseholdsReg /
                         bad_tbl.IndividualsReg.replace(0, np.nan)).round(2)
bad_tbl[['_id_site','dt','County','HouseholdsReg','IndividualsReg',
         'PoundsReg','site_ratio','overshoot']].sort_values('overshoot', ascending=False)

,_id_site,dt,County,HouseholdsReg,IndividualsReg,PoundsReg,site_ratio,overshoot
9567,1197.0,2023-07-01,Waseca,273385.0,646.0,12053.0,1.83,423.20
17376,1219.0,2024-11-01,Hennepin-Minneapolis,2202.0,98.0,94686.0,2.44,22.47
14179,15306.0,2024-04-01,Dakota,3257.0,371.0,3257.0,4.41,8.78
18123,15769.0,2024-06-01,Koochiching,176.0,38.0,2173.0,1.92,4.63
9145,14922.0,2023-06-01,Hennepin-Minneapolis,405.0,96.0,NaN,1.67,4.22
13094,14498.0,2024-02-01,Ramsey-Saint Paul,145.0,45.0,1125.0,1.00,3.22
9095,1245.0,2023-06-01,Polk,2247.0,706.0,36550.0,2.80,3.18
16259,1069.0,2024-09-01,Hennepin-Minneapolis,273.0,89.0,20291.0,1.54,3.07
10747,1307.0,2023-09-01,Wright,1157.0,454.0,10382.0,2.62,2.55
13096,14496.0,2024-02-01,Ramsey-Saint Paul,165.0,65.0,2340.0,1.00,2.54


In [13]:
mask = (fs2324._id_site == 1197.0) & (fs2324.dt == '2023-07-01')
fs2324.loc[mask, 'HouseholdsReg'] = 311

fs2324.loc[mask, ['_id_site','dt','County','HouseholdsReg','IndividualsReg','PoundsReg']]

,_id_site,dt,County,HouseholdsReg,IndividualsReg,PoundsReg
9567,1197.0,2023-07-01,Waseca,311.0,646.0,12053.0


In [14]:
mask.sum()                                    # 1
fs2324[fs2324.dt.dt.year == 2023].HouseholdsReg.sum()   # 2,517,551

np.float64(2517512.0)

In [15]:
fs2324 = (fs2324
    .sort_values(['_id_site','dt','IndividualsReg','PoundsReg','__ID'],
                 ascending=[True,True,False,False,False])
    .drop_duplicates(['_id_site','dt'], keep='first')
    .copy())                                     # <- this

mask = (fs2324._id_site == 1197.0) & (fs2324.dt == '2023-07-01')
fs2324['hh_imputed'] = False
fs2324.loc[mask, 'HouseholdsReg'] = 311
fs2324.loc[mask, 'hh_imputed'] = True

In [16]:
def ratio(df, key):
    g = df.groupby(key)[['IndividualsReg','HouseholdsReg']].sum()
    r = g.IndividualsReg / g.HouseholdsReg
    return r.where(r >= 1.5)          # below 1.5 is impossible as an average

site_r   = ratio(clean, '_id_site')
county_r = ratio(clean, 'County')
state_r  = clean.IndividualsReg.sum() / clean.HouseholdsReg.sum()     # 3.124

# rewrite only rows with a usable IndividualsReg, and never Waseca (set by hand)
fixable = bad & (fs2324.IndividualsReg > 0) & ~fs2324.hh_imputed      # 48
r = (fs2324._id_site.map(site_r)
            .fillna(fs2324.County.map(county_r))
            .fillna(state_r))

fs2324['hh_adjusted'] = bad          # all 53 suspect rows; clean bool, no .loc needed
fs2324.loc[fixable, 'HouseholdsReg'] = (fs2324.loc[fixable,'IndividualsReg'] / r[fixable]).round()

In [17]:
print(fs2324.hh_adjusted.sum())                       # 53  suspect as received
print(fs2324.hh_imputed.sum())                        # 1   Waseca, set by hand
print(fixable.sum())                                  # 48  rewritten from a ratio
print((bad & (fs2324.IndividualsReg == 0)).sum())     # 4   left alone -> decision 4
fs2324.groupby(fs2324.dt.dt.year).HouseholdsReg.sum() # 2023: 2506983  2024: 2900185

53
1
48
4


,HouseholdsReg
dt,
2023,2506983.0
2024,2900185.0


In [18]:
fs2324 = fs2324.drop(columns=['_Year','_Month'])
fs.DateForSearch = pd.to_datetime(fs.DateForSearch)

### D6
Three sites can be located from the roster; 39 cannot be located at all. Nothing is dropped: unlocatable rows stay in fsq1 behind a flag so county figures reconcile back to the source file.

In [ ]:

RECOVER = {15362.0: 'Rice',   # roster: Northfield, 55057, block_geoid 271310705032009
           15363.0: 'Rice',   # roster: Faribault,  55021, block_geoid 271310709011016
           15218.0: 'Cass'}   # roster: block_geoid 270219606001067 (files zeros all 18 months)

fsq1['county']            = fsq1.county.fillna(fsq1._id_site.map(RECOVER))
fsq1['county_fips']       = fsq1.county.map(fips)
fsq1['county_unassigned'] = fsq1.county.isna()
fsq1['year']              = fsq1.DateForSearch.dt.year

print(fsq1.county_unassigned.sum(),                               # 90
      fsq1.loc[fsq1.county_unassigned, '_id_site'].nunique())     # 39